# Credit Risk Scorecard — WOE / Logistic Regression

**Goal:** Build an interpretable credit risk scorecard using the methodology lenders and bureaus actually deploy under regulatory constraint — weight-of-evidence binning, information value feature selection, logistic regression, and points scaling — rather than a black-box classifier.

**Result:** Seven-variable scorecard achieving **AUC 0.857 / Gini 0.713 / KS 0.557** on a held-out test set, with no overfitting (test AUC marginally exceeds train). Default rates range from 54.2% in the lowest score band to 0.46% in the highest.

## Dataset
- Source: Kaggle "Give Me Some Credit" (2011 competition), `cs-training.csv`
- 150,000 borrowers, 10 predictors, binary target
- Target: `SeriousDlqin2yrs` — 90+ day delinquency within two years
- **Base rate: 6.68%.** Class imbalance means accuracy is meaningless; evaluation uses AUC, Gini, and KS.

## Why WOE rather than raw features

Weight of Evidence transforms each bin into `ln(good% / bad%)`, which linearizes non-linear relationships, handles missing values as their own category, and neutralizes outliers. The resulting model is fully explainable — a requirement under adverse action notice rules, since a declined applicant must be told which factors drove the decision.

## Data quality findings

Three problems were found by inspection. Each was investigated before treatment, and the correct treatment differed in each case.

**1. Sentinel codes (96, 98) in delinquency variables.** No borrower is 98 times 60–89 days past due. These are placeholder codes from the source system, likely encoding account states (charge-off, collections) where a delinquency counter no longer applies. The same 269 records carry the code across all three delinquency columns.

**Their default rate is 54.65% vs. 6.68% overall — 8x the base rate.** Dropping them as errors would have discarded the single strongest signal in the dataset. Retained and binned separately. Empirically their WOE (−2.82) sits between the "2 lates" and "3+ lates" bins, which partially identifies what the codes encode.

**2. Extreme utilization values.** `RevolvingUtilizationOfUnsecuredLines` reaches 50,708 — a mathematically impossible ratio. Investigation by band:

| Utilization | n | Default rate |
|---|---|---|
| 0–1 | 146,662 | 5.99% |
| 1–2 | 2,967 | 40.01% |
| 2–10 | 130 | 28.46% |
| 10+ | 241 | 7.05% |

Values in the 1–2 range are legitimate and highly predictive (over-limit borrowers). Values above 10 default at the base rate — they carry no signal and are corrupted denominators. Opposite conclusion from the sentinel codes, reached by the same investigation.

**3. DebtRatio as a disguised proxy.** `DebtRatio` reaches 329,664. 28,877 borrowers exceed 10 — and **26,771 of them (93%) are the same records missing `MonthlyIncome`.** With income missing or zero, the ratio is uncomputable and the field was populated with raw debt or a fallback. Their default rate (5.57%) matches the missing-income population (5.61%) rather than any debt-burden pattern. Binned as "not meaningful" rather than capped, so the remaining bins measure actual debt burden.

## Missingness is informative

`MonthlyIncome` is missing for 29,731 borrowers (19.8%). These borrowers default at **5.61% vs. 6.95%** for those with income reported — missing income is associated with *lower* risk, not higher. Missing values are retained as their own bin rather than imputed or dropped, consistent with scorecard practice.

## Feature selection

| Variable | IV | Decision |
|---|---|---|
| Revolving utilization | 1.133 | Retained |
| 90+ days late | 0.878 | Retained |
| 30–59 days late | 0.758 | Retained |
| 60–89 days late | 0.600 | Retained |
| Age | 0.256 | Retained |
| Open credit lines | 0.080 | **Dropped** — non-monotonic (U-shaped) |
| Monthly income | 0.076 | Retained (significant at p=0.003) |
| Real estate loans | 0.062 | **Dropped** — non-monotonic |
| Debt ratio | 0.061 | Retained |
| Dependents | 0.036 | **Dropped** — below IV threshold |

IV convention: <0.02 useless, 0.02–0.10 weak, 0.10–0.30 medium, 0.30–0.50 strong, >0.50 verify for leakage.

**Leakage check.** Utilization (1.13) and 90-day delinquency (0.88) both exceed the 0.50 threshold. Both are benign: current credit utilization and prior payment history are the two most established predictors in consumer credit and are legitimately available at scoring time. No forward-looking information is present.

**Binning judgment.** Two bins were merged to enforce monotonicity: under-25 with 25–30 (age showed a reversal at the young end), and utilization 2.0+ with 1.0–2.0 (the corrupted tail pulled WOE back toward average). Both merges cost negligible IV (0.001 and 0.009) and eliminated reversals that would be indefensible in review. Non-monotonic bins are a review failure even when statistically real.

## Multicollinearity

Pairwise correlation among the three delinquency counts is 0.22–0.31 — lower than expected. The variables are not nested; they count distinct events at different severity levels, so a borrower who was repeatedly 30 days late but always caught up before 60 registers only in one column.

VIF after WOE transformation: all seven features between **1.08 and 1.24**, well below the 5.0 concern threshold. All retained.

## Model

Logistic regression on WOE-transformed features, 70/30 stratified split (105,000 train / 45,000 test, 6.68% default rate in both).

| Feature | Coefficient | p-value |
|---|---|---|
| Utilization | −0.626 | <0.001 |
| 90+ days late | −0.550 | <0.001 |
| 30–59 days late | −0.471 | <0.001 |
| 60–89 days late | −0.372 | <0.001 |
| Age | −0.428 | <0.001 |
| Income | −0.155 | 0.003 |
| Debt ratio | −0.800 | <0.001 |
| Intercept | −2.604 | <0.001 |

Pseudo R² = 0.2515. All coefficients are negative and consistently signed — correct, since higher WOE indicates lower risk while the target codes 1 as default. No sign flips.

Note that debt ratio carries the largest coefficient despite the second-weakest IV. IV measures standalone predictive power; the coefficient measures marginal contribution after controlling for other variables. Debt ratio is weak alone but adds information the others don't.

## Validation

| Metric | Train | Test |
|---|---|---|
| AUC | 0.8557 | 0.8565 |
| Gini | 0.7114 | 0.7131 |

**KS statistic (test): 0.5573**

Test performance marginally exceeds train, indicating no overfitting. Coarse binning removes the noise a more flexible model would memorize. Production scorecards typically achieve Gini in the 0.40–0.60 range; KS above 0.40 is considered strong.

The relevant claim is not that this beats a gradient boosting model — it likely wouldn't. It is that a seven-variable, fully explainable model reaches 0.86 AUC, and explainability is a deployment prerequisite rather than a preference.

## Scorecard scaling

Points scaled with base score 600 at 50:1 odds, PDO 20 (20 points doubles the odds of repayment).

**Scale parameters are conventional but arbitrary — these scores are not comparable to FICO.** Changing the base score or PDO relabels every score without altering the model.

Point ranges by variable indicate practical influence:
- Utilization: 42–105 (63-point swing — widest)
- 30–59 days late: 42–88
- 90+ days late: 31–87
- 60–89 days late: 48–83
- Age: 73–93
- Debt ratio: 68–85
- Income: 79–83 (4-point swing — statistically significant, practically negligible)

## Score band performance

Scored on a 20,000-borrower sample:

| Score band | Borrowers | Default rate |
|---|---|---|
| ≤500 | 627 | 54.23% |
| 500–550 | 2,079 | 21.69% |
| 550–600 | 9,679 | 4.65% |
| 600–620 | 6,526 | 0.83% |
| 620+ | 1,089 | 0.46% |

Monotonic across all bands, with a **118x spread** between lowest and highest. A cutoff at 550 would decline 13.5% of applicants and avoid roughly half of all defaults.

## Limitations

- **Vintage.** Data reflects 2011 lending conditions. The methodology is period-independent, but the coefficients are not; a production model would be rebuilt on current data and monitored for population stability (PSI).
- **Score distribution is compressed at the top.** Median 592, max 623, with 75% of borrowers above 565. The scorecard separates good from bad well but discriminates poorly *among* good borrowers. A production build would use finer bins in the high range.
- **No out-of-time validation.** The train/test split is random, not temporal. A real model risk review requires validation on a later time period to test stability, which this dataset does not support.
- **Redundant encoding.** The DebtRatio "not meaningful" bin and the income "Missing" bin encode the same underlying population (WOE +0.194 and +0.186 respectively). Both were retained since VIF showed no multicollinearity issue, but one is arguably duplicative.
- **Reject inference not performed.** This data contains only booked accounts. Production scorecards must account for applicants who were declined and therefore have no performance outcome, or the model is biased toward the accepted population.
- **No fairness testing.** Disparate impact analysis across protected classes was not performed; the dataset contains no demographic fields beyond age. Any deployed model requires fair lending review.

## Next steps
- Reject inference simulation
- Population stability monitoring framework (PSI)
- Fairness/disparate impact analysis on a dataset with demographic fields
- Comparison against gradient boosting to quantify the interpretability trade-off

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Data/cs-training.csv", index_col=0)
print(df.shape)

df.head()

(150000, 11)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [2]:
print("Rows:", len(df))
print("\nDefault rate:", round(100 * df["SeriousDlqin2yrs"].mean(), 2), "%")
print("\nMissing values:")
print(df.isnull().sum())

Rows: 150000

Default rate: 6.68 %

Missing values:
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64


In [3]:
print("Default rate, income MISSING:", 
      round(100 * df[df["MonthlyIncome"].isnull()]["SeriousDlqin2yrs"].mean(), 2), "%")
print("Default rate, income PRESENT:", 
      round(100 * df[df["MonthlyIncome"].notnull()]["SeriousDlqin2yrs"].mean(), 2), "%")

Default rate, income MISSING: 5.61 %
Default rate, income PRESENT: 6.95 %


In [4]:
for col in ["NumberOfTimes90DaysLate", "NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTime60-89DaysPastDueNotWorse"]:
    print(f"\n=== {col} ===")
    print(df[col].value_counts().sort_index().head(15))


=== NumberOfTimes90DaysLate ===
NumberOfTimes90DaysLate
0     141662
1       5243
2       1555
3        667
4        291
5        131
6         80
7         38
8         21
9         19
10         8
11         5
12         2
13         4
14         2
Name: count, dtype: int64

=== NumberOfTime30-59DaysPastDueNotWorse ===
NumberOfTime30-59DaysPastDueNotWorse
0     126018
1      16033
2       4598
3       1754
4        747
5        342
6        140
7         54
8         25
9         12
10         4
11         1
12         2
13         1
96         5
Name: count, dtype: int64

=== NumberOfTime60-89DaysPastDueNotWorse ===
NumberOfTime60-89DaysPastDueNotWorse
0     142396
1       5731
2       1118
3        318
4        105
5         34
6         16
7          9
8          2
9          1
11         1
96         5
98       264
Name: count, dtype: int64


In [5]:
weird = df[(df["NumberOfTime30-59DaysPastDueNotWorse"] >= 96) | 
           (df["NumberOfTime60-89DaysPastDueNotWorse"] >= 96) |
           (df["NumberOfTimes90DaysLate"] >= 96)]

print("Rows with sentinel codes:", len(weird))
print("Default rate for these rows:", round(100 * weird["SeriousDlqin2yrs"].mean(), 2), "%")
print("Default rate overall:", round(100 * df["SeriousDlqin2yrs"].mean(), 2), "%")

Rows with sentinel codes: 269
Default rate for these rows: 54.65 %
Default rate overall: 6.68 %


In [6]:
tab = df.groupby("NumberOfTimes90DaysLate")["SeriousDlqin2yrs"].agg(["count", "mean"])
tab["default_rate_%"] = (100 * tab["mean"]).round(2)
print(tab[["count", "default_rate_%"]].head(12))

                          count  default_rate_%
NumberOfTimes90DaysLate                        
0                        141662            4.63
1                          5243           33.66
2                          1555           49.90
3                           667           57.72
4                           291           67.01
5                           131           63.36
6                            80           60.00
7                            38           81.58
8                            21           71.43
9                            19           73.68
10                            8           62.50
11                            5           60.00


In [7]:
for col in ["NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTime60-89DaysPastDueNotWorse"]:
    tab = df[df[col] < 90].groupby(col)["SeriousDlqin2yrs"].agg(["count", "mean"])
    tab["default_rate_%"] = (100 * tab["mean"]).round(2)
    print(f"\n=== {col} ===")
    print(tab[["count", "default_rate_%"]].head(8))


=== NumberOfTime30-59DaysPastDueNotWorse ===
                                       count  default_rate_%
NumberOfTime30-59DaysPastDueNotWorse                        
0                                     126018            4.00
1                                      16033           15.03
2                                       4598           26.51
3                                       1754           35.23
4                                        747           42.57
5                                        342           45.03
6                                        140           52.86
7                                         54           51.85

=== NumberOfTime60-89DaysPastDueNotWorse ===
                                       count  default_rate_%
NumberOfTime60-89DaysPastDueNotWorse                        
0                                     142396            5.10
1                                       5731           31.01
2                                       1118          

In [8]:
bins = [0, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 120]
df["age_bin"] = pd.cut(df["age"], bins=bins)

tab = df.groupby("age_bin", observed=True)["SeriousDlqin2yrs"].agg(["count", "mean"])
tab["default_rate_%"] = (100 * tab["mean"]).round(2)
print(tab[["count", "default_rate_%"]])

           count  default_rate_%
age_bin                         
(0, 25]     3027           11.17
(25, 30]    7730           11.72
(30, 35]   10728           10.69
(35, 40]   13611            9.13
(40, 45]   16208            8.55
(45, 50]   18829            8.01
(50, 55]   17861            7.16
(55, 60]   16945            5.14
(60, 65]   16461            4.01
(65, 70]   10963            2.66
(70, 120]  17636            2.26


In [9]:
def calc_woe(df, feature, target="SeriousDlqin2yrs"):
    grouped = df.groupby(feature, observed=True)[target].agg(["count", "sum"])
    grouped.columns = ["total", "bad"]
    grouped["good"] = grouped["total"] - grouped["bad"]
    
    grouped["bad_pct"] = grouped["bad"] / grouped["bad"].sum()
    grouped["good_pct"] = grouped["good"] / grouped["good"].sum()
    
    grouped["woe"] = np.log(grouped["good_pct"] / grouped["bad_pct"])
    grouped["iv_contrib"] = (grouped["good_pct"] - grouped["bad_pct"]) * grouped["woe"]
    
    print(f"Information Value: {grouped['iv_contrib'].sum():.4f}")
    return grouped[["total", "bad", "default_rate" ] if False else ["total", "bad", "woe", "iv_contrib"]].round(4)

calc_woe(df, "age_bin")

Information Value: 0.2569


,total,bad,woe,iv_contrib
age_bin,,,,
"(0, 25]",3027,338,-0.5624,0.0082
"(25, 30]",7730,906,-0.6171,0.0257
"(30, 35]",10728,1147,-0.5136,0.0236
"(35, 40]",13611,1243,-0.3387,0.0121
"(40, 45]",16208,1385,-0.2658,0.0086
"(45, 50]",18829,1508,-0.1951,0.0052
"(50, 55]",17861,1278,-0.0732,0.0007
"(55, 60]",16945,871,0.2790,0.0078
"(60, 65]",16461,660,0.5393,0.0254


In [10]:
def bin_delinq(x):
    if x >= 90: return "sentinel(96/98)"
    if x == 0: return "0"
    if x == 1: return "1"
    if x == 2: return "2"
    return "3+"

df["late90_bin"] = df["NumberOfTimes90DaysLate"].apply(bin_delinq)
calc_woe(df, "late90_bin")

Information Value: 0.8780


,total,bad,woe,iv_contrib
late90_bin,,,,
0,141662,6554,0.3897,0.1214
1,5243,1765,-1.9580,0.2960
2,1555,776,-2.6324,0.1891
3+,1271,784,-3.1124,0.2326
sentinel(96/98),269,147,-2.8227,0.0389


In [11]:
df["late30_bin"] = df["NumberOfTime30-59DaysPastDueNotWorse"].apply(bin_delinq)
df["late60_bin"] = df["NumberOfTime60-89DaysPastDueNotWorse"].apply(bin_delinq)

print("=== 30-59 days ===")
display(calc_woe(df, "late30_bin"))

print("=== 60-89 days ===")
display(calc_woe(df, "late60_bin"))

=== 30-59 days ===
Information Value: 0.7575


,total,bad,woe,iv_contrib
late30_bin,,,,
0,126018,5041,0.5417,0.1958
1,16033,2409,-0.9037,0.1292
2,4598,1219,-1.6167,0.1575
3+,3082,1210,-2.1999,0.2361
sentinel(96/98),269,147,-2.8227,0.0389


=== 60-89 days ===
Information Value: 0.6002


,total,bad,woe,iv_contrib
late60_bin,,,,
0,142396,7256,0.2882,0.0697
1,5731,1777,-1.8365,0.2736
2,1118,561,-2.6434,0.1374
3+,486,285,-2.9855,0.0806
sentinel(96/98),269,147,-2.8227,0.0389


In [12]:
delinq_cols = ["NumberOfTime30-59DaysPastDueNotWorse", 
               "NumberOfTime60-89DaysPastDueNotWorse",
               "NumberOfTimes90DaysLate"]

clean = df[df[delinq_cols].max(axis=1) < 90]
print(clean[delinq_cols].corr().round(3))

                                      NumberOfTime30-59DaysPastDueNotWorse  \
NumberOfTime30-59DaysPastDueNotWorse                                 1.000   
NumberOfTime60-89DaysPastDueNotWorse                                 0.306   
NumberOfTimes90DaysLate                                              0.218   

                                      NumberOfTime60-89DaysPastDueNotWorse  \
NumberOfTime30-59DaysPastDueNotWorse                                 0.306   
NumberOfTime60-89DaysPastDueNotWorse                                 1.000   
NumberOfTimes90DaysLate                                              0.295   

                                      NumberOfTimes90DaysLate  
NumberOfTime30-59DaysPastDueNotWorse                    0.218  
NumberOfTime60-89DaysPastDueNotWorse                    0.295  
NumberOfTimes90DaysLate                                 1.000  


In [13]:
print(df["RevolvingUtilizationOfUnsecuredLines"].describe())
print("\nAbove 1.0:", (df["RevolvingUtilizationOfUnsecuredLines"] > 1).sum())
print("Above 10:", (df["RevolvingUtilizationOfUnsecuredLines"] > 10).sum())

count    150000.000000
mean          6.048438
std         249.755371
min           0.000000
25%           0.029867
50%           0.154181
75%           0.559046
max       50708.000000
Name: RevolvingUtilizationOfUnsecuredLines, dtype: float64

Above 1.0: 3321
Above 10: 241


In [14]:
for lo, hi, label in [(0, 1, "0-1 (normal)"), (1, 2, "1-2"), (2, 10, "2-10"), (10, 1e9, "10+")]:
    sub = df[(df["RevolvingUtilizationOfUnsecuredLines"] >= lo) & 
             (df["RevolvingUtilizationOfUnsecuredLines"] < hi)]
    print(f"{label:15} n={len(sub):>7}  default={100*sub['SeriousDlqin2yrs'].mean():.2f}%")

0-1 (normal)    n= 146662  default=5.99%
1-2             n=   2967  default=40.01%
2-10            n=    130  default=28.46%
10+             n=    241  default=7.05%


In [15]:
util_bins = [-0.01, 0.1, 0.3, 0.5, 0.75, 1.0, 1e9]
util_labels = ["0-0.1", "0.1-0.3", "0.3-0.5", "0.5-0.75", "0.75-1.0", "1.0+"]
df["util_bin"] = pd.cut(df["RevolvingUtilizationOfUnsecuredLines"], bins=util_bins, labels=util_labels)

calc_woe(df, "util_bin")

Information Value: 1.1333


,total,bad,woe,iv_contrib
util_bin,,,,
0-0.1,64404,1166,1.3571,0.4553
0.1-0.3,28478,895,0.7919,0.0854
0.3-0.5,15830,926,0.1422,0.0020
0.5-0.75,13764,1394,-0.4532,0.0230
0.75-1.0,24203,4408,-1.1343,0.3383
1.0+,3321,1237,-2.1147,0.2294


In [16]:
def bin_income(x):
    if pd.isnull(x): return "Missing"
    if x == 0: return "0"
    if x < 2000: return "1-2k"
    if x < 4000: return "2-4k"
    if x < 6000: return "4-6k"
    if x < 8000: return "6-8k"
    if x < 12000: return "8-12k"
    return "12k+"

df["income_bin"] = df["MonthlyIncome"].apply(bin_income)
calc_woe(df, "income_bin")

Information Value: 0.0763


,total,bad,woe,iv_contrib
income_bin,,,,
0,1634,66,0.5316,0.0025
1-2k,9253,848,-0.3426,0.0084
12k+,11289,509,0.4167,0.0109
2-4k,27405,2549,-0.3589,0.0275
4-6k,28848,2140,-0.1121,0.0025
6-8k,20551,1241,0.1084,0.0015
8-12k,21289,1004,0.3696,0.0166
Missing,29731,1669,0.1859,0.0063


In [17]:
def bin_count(x, cuts):
    for c in cuts:
        if x <= c: return f"<={c}"
    return f">{cuts[-1]}"

df["credlines_bin"] = df["NumberOfOpenCreditLinesAndLoans"].apply(lambda x: bin_count(x, [2, 4, 6, 8, 12, 18]))
df["realestate_bin"] = df["NumberRealEstateLoansOrLines"].apply(lambda x: bin_count(x, [0, 1, 2, 3]))
df["depend_bin"] = df["NumberOfDependents"].apply(lambda x: "Missing" if pd.isnull(x) else bin_count(x, [0, 1, 2, 3]))

for c in ["credlines_bin", "realestate_bin", "depend_bin"]:
    print(f"\n=== {c} ===")
    display(calc_woe(df, c))


=== credlines_bin ===
Information Value: 0.0801


,total,bad,woe,iv_contrib
credlines_bin,,,,
<=12,36305,2151,0.1287,0.0038
<=18,21102,1387,0.0180,0.0000
<=2,12992,1678,-0.7278,0.0630
<=4,20667,1425,-0.0334,0.0002
<=6,26545,1573,0.1285,0.0028
<=8,25807,1353,0.2582,0.0103
>18,6582,459,-0.0455,0.0001



=== realestate_bin ===
Information Value: 0.0617


,total,bad,woe,iv_contrib
realestate_bin,,,,
<=0,56188,4672,-0.2360,0.0231
<=1,52338,2748,0.2566,0.0206
<=2,31522,1765,0.1886,0.0069
<=3,6300,422,-0.0023,0.0000
>3,3652,419,-0.5930,0.0111



=== depend_bin ===
Information Value: 0.0359


,total,bad,woe,iv_contrib
depend_bin,,,,
<=0,86902,5095,0.1398,0.0107
<=1,26316,1935,-0.1026,0.0019
<=2,19522,1584,-0.2093,0.0062
<=3,9483,837,-0.3012,0.0065
>3,3853,396,-0.4695,0.0069
Missing,3924,179,0.4045,0.0036


In [18]:
print(df["DebtRatio"].describe())
print("\nAbove 1:", (df["DebtRatio"] > 1).sum())
print("Above 10:", (df["DebtRatio"] > 10).sum())

count    150000.000000
mean        353.005076
std        2037.818523
min           0.000000
25%           0.175074
50%           0.366508
75%           0.868254
max      329664.000000
Name: DebtRatio, dtype: float64

Above 1: 35137
Above 10: 28877


In [19]:
high_dr = df["DebtRatio"] > 10
missing_inc = df["MonthlyIncome"].isnull()

print("DebtRatio > 10:", high_dr.sum())
print("Missing income:", missing_inc.sum())
print("Both:", (high_dr & missing_inc).sum())
print("\nDefault rate, DebtRatio > 10:", round(100 * df[high_dr]["SeriousDlqin2yrs"].mean(), 2), "%")
print("Default rate, DebtRatio <= 10:", round(100 * df[~high_dr]["SeriousDlqin2yrs"].mean(), 2), "%")

DebtRatio > 10: 28877
Missing income: 29731
Both: 26771

Default rate, DebtRatio > 10: 5.57 %
Default rate, DebtRatio <= 10: 6.95 %


In [20]:
def bin_debtratio(row):
    dr = row["DebtRatio"]
    if dr > 10: return "Not meaningful (income missing/zero)"
    if dr <= 0.1: return "0-0.1"
    if dr <= 0.3: return "0.1-0.3"
    if dr <= 0.5: return "0.3-0.5"
    if dr <= 0.75: return "0.5-0.75"
    if dr <= 1.0: return "0.75-1.0"
    return "1.0-10"

df["debtratio_bin"] = df.apply(bin_debtratio, axis=1)
calc_woe(df, "debtratio_bin")

Information Value: 0.0607


,total,bad,woe,iv_contrib
debtratio_bin,,,,
0-0.1,24520,1454,0.1278,0.0025
0.1-0.3,38041,2220,0.1448,0.0050
0.3-0.5,31147,1981,0.0531,0.0006
0.5-0.75,15728,1484,-0.3747,0.0173
0.75-1.0,5427,596,-0.5437,0.0136
1.0-10,6260,682,-0.5347,0.0151
Not meaningful (income missing/zero),28877,1609,0.1938,0.0067


In [21]:
def get_woe_map(df, feature, target="SeriousDlqin2yrs"):
    g = df.groupby(feature, observed=True)[target].agg(["count", "sum"])
    g.columns = ["total", "bad"]
    g["good"] = g["total"] - g["bad"]
    g["woe"] = np.log((g["good"] / g["good"].sum()) / (g["bad"] / g["bad"].sum()))
    iv = (((g["good"]/g["good"].sum()) - (g["bad"]/g["bad"].sum())) * g["woe"]).sum()
    return g["woe"].to_dict(), iv

features = ["util_bin", "late90_bin", "late30_bin", "late60_bin", 
            "age_bin", "income_bin", "debtratio_bin"]

woe_maps = {}
for f in features:
    woe_maps[f], iv = get_woe_map(df, f)
    df[f + "_woe"] = df[f].map(woe_maps[f])
    print(f"{f:20} IV={iv:.4f}")

woe_features = [f + "_woe" for f in features]
print("\nMissing after mapping:", df[woe_features].isnull().sum().sum())

util_bin             IV=1.1333
late90_bin           IV=0.8780
late30_bin           IV=0.7575
late60_bin           IV=0.6002
age_bin              IV=0.2569
income_bin           IV=0.0763
debtratio_bin        IV=0.0607

Missing after mapping: 1


In [22]:
features = ["util_bin", "late90_bin", "late30_bin", "late60_bin", 
            "age_bin", "income_bin", "debtratio_bin"]

woe_maps = {}
for f in features:
    woe_maps[f], iv = get_woe_map(df, f)
    df[f + "_woe"] = df[f].map(woe_maps[f])
    print(f"{f:20} IV={iv:.4f}")

woe_features = [f + "_woe" for f in features]
print("\nMissing after mapping:", df[woe_features].isnull().sum().sum())

util_bin             IV=1.1333
late90_bin           IV=0.8780
late30_bin           IV=0.7575
late60_bin           IV=0.6002
age_bin              IV=0.2569
income_bin           IV=0.0763
debtratio_bin        IV=0.0607

Missing after mapping: 1


In [23]:
for f in features:
    n = df[f + "_woe"].isnull().sum()
    if n > 0:
        print(f"{f}: {n} missing")
        print(df[df[f + "_woe"].isnull()][f].unique())

age_bin: 1 missing
[NaN]
Categories (11, interval[int64, right]): [(0, 25] < (25, 30] < (30, 35] < (35, 40] ... (55, 60] < (60, 65] < (65, 70] < (70, 120]]


In [24]:
print("Rows with age 0:", (df["age"] == 0).sum())

df["age_bin"] = pd.cut(df["age"], bins=[-1, 30, 35, 40, 45, 50, 55, 60, 65, 120])
woe_maps["age_bin"], iv_age = get_woe_map(df, "age_bin")
df["age_bin_woe"] = df["age_bin"].map(woe_maps["age_bin"])

print("Age IV after merge:", round(iv_age, 4))
print("Missing now:", df[woe_features].isnull().sum().sum())

Rows with age 0: 1
Age IV after merge: 0.2559
Missing now: 0


# Dependencies: pip install scikit-learn statsmodels

In [25]:
from sklearn.model_selection import train_test_split

X = df[woe_features]
y = df["SeriousDlqin2yrs"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Train:", X_train.shape, "| default rate:", round(100*y_train.mean(), 2), "%")
print("Test: ", X_test.shape, "| default rate:", round(100*y_test.mean(), 2), "%")

Train: (105000, 7) | default rate: 6.68 %
Test:  (45000, 7) | default rate: 6.68 %


In [26]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

Xc = sm.add_constant(X_train)
vif = pd.DataFrame({
    "feature": Xc.columns,
    "VIF": [variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])]
})
print(vif.round(2))

             feature   VIF
0              const  1.35
1       util_bin_woe  1.24
2     late90_bin_woe  1.24
3     late30_bin_woe  1.21
4     late60_bin_woe  1.23
5        age_bin_woe  1.08
6     income_bin_woe  1.08
7  debtratio_bin_woe  1.09


In [27]:
import statsmodels.api as sm

X_train_c = sm.add_constant(X_train)
model = sm.Logit(y_train, X_train_c).fit(disp=0)
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:       SeriousDlqin2yrs   No. Observations:               105000
Model:                          Logit   Df Residuals:                   104992
Method:                           MLE   Df Model:                            7
Date:                Sat, 22 Aug 2026   Pseudo R-squ.:                  0.2515
Time:                        20:19:21   Log-Likelihood:                -19284.
converged:                       True   LL-Null:                       -25765.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -2.6038      0.015   -173.271      0.000      -2.633      -2.574
util_bin_woe         -0.6260      0.014    -44.059      0.000      -0.654      -0.598
late90_bin_woe       -0.

In [28]:
from sklearn.metrics import roc_auc_score, roc_curve

X_test_c = sm.add_constant(X_test)
pred_train = model.predict(X_train_c)
pred_test = model.predict(X_test_c)

auc_train = roc_auc_score(y_train, pred_train)
auc_test = roc_auc_score(y_test, pred_test)

fpr, tpr, _ = roc_curve(y_test, pred_test)
ks = max(tpr - fpr)

print(f"AUC train: {auc_train:.4f}  (Gini {2*auc_train-1:.4f})")
print(f"AUC test:  {auc_test:.4f}  (Gini {2*auc_test-1:.4f})")
print(f"KS statistic (test): {ks:.4f}")

AUC train: 0.8557  (Gini 0.7114)
AUC test:  0.8565  (Gini 0.7131)
KS statistic (test): 0.5573


In [29]:
# Standard scaling: 600 points at 50:1 odds, 20 points to double the odds
base_score = 600
base_odds = 50
pdo = 20

factor = pdo / np.log(2)
offset = base_score - factor * np.log(base_odds)

print(f"Factor: {factor:.4f}, Offset: {offset:.4f}")

# points per bin
n_vars = len(woe_features)
intercept = model.params["const"]

scorecard_rows = []
for f in features:
    coef = model.params[f + "_woe"]
    for bin_name, woe in woe_maps[f].items():
        points = -(coef * woe + intercept / n_vars) * factor + offset / n_vars
        scorecard_rows.append({
            "variable": f,
            "bin": str(bin_name),
            "woe": round(woe, 4),
            "points": round(points)
        })

scorecard = pd.DataFrame(scorecard_rows)
print(scorecard.to_string(index=False))

Factor: 28.8539, Offset: 487.1229
     variable                                  bin     woe  points
     util_bin                                0-0.1  1.3571     105
     util_bin                              0.1-0.3  0.7919      95
     util_bin                              0.3-0.5  0.1422      83
     util_bin                             0.5-0.75 -0.4532      72
     util_bin                             0.75-1.0 -1.1343      60
     util_bin                                 1.0+ -2.1147      42
   late90_bin                                    0  0.3897      87
   late90_bin                                    1 -1.9580      49
   late90_bin                                    2 -2.6324      39
   late90_bin                                   3+ -3.1124      31
   late90_bin                      sentinel(96/98) -2.8227      36
   late30_bin                                    0  0.5417      88
   late30_bin                                    1 -0.9037      68
   late30_bin               

In [30]:
def score_row(row):
    total = 0
    for f in features:
        coef = model.params[f + "_woe"]
        woe = woe_maps[f][row[f]]
        total += -(coef * woe + intercept / n_vars) * factor + offset / n_vars
    return total

sample = df.sample(20000, random_state=42).copy()
sample["score"] = sample.apply(score_row, axis=1)

print(sample["score"].describe().round(0))

bands = pd.cut(sample["score"], bins=[0, 500, 550, 600, 620, 640, 1000])
tab = sample.groupby(bands, observed=True)["SeriousDlqin2yrs"].agg(["count", "mean"])
tab["default_rate_%"] = (100 * tab["mean"]).round(2)
print("\n", tab[["count", "default_rate_%"]])

count    20000.0
mean       582.0
std         34.0
min        408.0
25%        565.0
50%        592.0
75%        606.0
max        623.0
Name: score, dtype: float64

             count  default_rate_%
score                            
(0, 500]      627           54.23
(500, 550]   2079           21.69
(550, 600]   9679            4.65
(600, 620]   6526            0.83
(620, 640]   1089            0.46
